# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup: connect to the warehouse

This notebook runs against the full Hugging Face warehouse release, not the small starter CSV.
Before running: on the [dataset page](https://huggingface.co/datasets/FlyRank/internship-warehouse)
click "request access" (instant), then create a plain **Read** token in your HF settings
(tick the gated-repositories permission) and store it as a Colab Secret named `HF_TOKEN`.


In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# mid-panel month, per the card's warning: never develop label logic on the sealed final month
MONTH_START = "2026-03-01"
MONTH_END   = "2026-03-31"
DECISION_DAY = "2026-03-15"  # splits the month into a feature half and a target half
print("connected. working month:", MONTH_START, "to", MONTH_END)


connected. working month: 2026-03-01 to 2026-03-31


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content item (`content_hash_id`) belonging to one client
(`client_hash_id`), aggregated over a single month of daily search/engagement activity from
`fact_content_daily_performance`.

**Time window:** March 2026 (`report_date` between `2026-03-01` and `2026-03-31`) — a mid-panel
month, not the sealed final month (`2026-06`, which is what `_sample` points at). Within this
month I split at `2026-03-15`: days 1–15 are the **feature window** (what I'm allowed to know at
the decision moment), days 16–31 are the **target window** (what I'm trying to say something
about). This mirrors the prior-90-days → next-30-days structure I'll need for the real capstone
label, just compressed into one month for this week's exercise.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (known by the decision moment, day 15):**
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` — summed/averaged over days 1–15
- `ga4_sessions` — summed over days 1–15
- `dim_content.content_created_at` — fixed at content creation, always known

**Label / proxy:**
- `is_declining` = impressions in days 16–31 fell more than 20% below impressions in days 1–15.
  This is a proxy for "this page is losing visibility," measured from an outcome the feature
  window did not see.

**Context (useful for interpretation, not fed to any model):**
- `client_hash_id`, `content_hash_id` — join keys only, carry no signal themselves
- `dim_clients.gsc_data_start` / `ga4_data_start` — tells me if a client's history even covers
  this month cleanly

**Excluded, on purpose:**
- `ga4_data_available` — I filter on it (see query 3) but never feed it in as a feature; it's a
  data-quality flag, not a search or content signal, and including it risks the model just
  learning "rows with less tracking behave differently" instead of anything about the content.
- Any raw query/URL/title text — not in this release at all, and wouldn't be public-safe if it were.


## 3. Verify it with queries (grain, counts, missing values, windows) — five features — the trap

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — grain: one row really is one (client, content, date)


In [3]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()

print("rows that violate one-row-per-(client,content,date):", len(grain_check))
grain_check.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows that violate one-row-per-(client,content,date): 0


,client_hash_id,content_hash_id,report_date,n


### Query 2 — my slice's row count and date span

In [4]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_date, MAX(report_date) AS last_date,
           COUNT(DISTINCT content_hash_id) AS n_content, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
""").df()
span


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,first_date,last_date,n_content,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


### Query 3 — availability, filtered with `IS TRUE`

`ga4_data_available` marks rows from before a client's GA4 tracking started (search-only rows).
I check how many March rows actually carry usable engagement data.


In [5]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_march_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
""").df()

availability["pct_available"] = availability["ga4_available_rows"] / availability["total_march_rows"]
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,ga4_available_rows,pct_available
0,9841378,413966,0.042064


### Five features (max) — built from the feature window only (days 1–15)

Each feature: knowable at the decision moment because it only uses `report_date <= DECISION_DAY`.


In [7]:
feature_frame = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)                                AS impressions_first_half,
        SUM(f.gsc_clicks)                                      AS clicks_first_half,
        AVG(f.gsc_avg_position)                                AS avg_position_first_half,
        SUM(f.ga4_sessions)                                    AS sessions_first_half,
        DATE_DIFF('day', c.content_created_date, DATE '{DECISION_DAY}') AS content_age_days_at_decision
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date BETWEEN '{MONTH_START}' AND '{DECISION_DAY}'
    GROUP BY 1, 2, c.content_created_date
    HAVING impressions_first_half >= 50
""").df()

print(f"{len(feature_frame):,} content items with enough first-half signal")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 content items with enough first-half signal


,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,sessions_first_half,content_age_days_at_decision
0,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,66.0,0.0,12.282875,NaN,159
1,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,440.0,2.0,11.741716,NaN,159
2,client_0797ff3a1fc9a6a5,content_9d5a482ec16ea617,54.0,1.0,9.820068,NaN,159
3,client_0797ff3a1fc9a6a5,content_a7a5f0d74ec03ce8,1621.0,14.0,11.086430,NaN,159
4,client_0797ff3a1fc9a6a5,content_be06033d30b49299,1356.0,0.0,56.326946,NaN,159


**Why each is knowable at the decision moment (day 15):**
1. `impressions_first_half` — summed only over days already elapsed by day 15.
2. `clicks_first_half` — same window restriction, same reasoning.
3. `avg_position_first_half` — averaged only over days 1–15; position on day 20 isn't touched.
4. `sessions_first_half` — GA4 sessions, same day-15 cutoff.
5. `content_age_days_at_decision` — computed from `content_created_at`, which is fixed at
   publish time and never depends on anything in the target window.


### The trap: add a label-derived column on purpose, watch the score jump, then remove it

Label: `is_declining` = impressions in the **target** window (days 16–31) fell more than 20%
below `impressions_first_half`. I build it from the target window on purpose here — that's what
makes the next leaky feature a genuine trap, not a hypothetical one.


In [8]:
labels = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_second_half
    FROM {TABLES['fact_daily']}
    WHERE report_date > DATE '{DECISION_DAY}' AND report_date <= DATE '{MONTH_END}'
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(labels, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["impressions_second_half"] < 0.8 * data["impressions_first_half"]).astype(int)
print(f"{len(data):,} rows with both halves present, positive rate: {data['is_declining'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 rows with both halves present, positive rate: 28.6%


**Honest score first — features from the feature window only:**

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_first_half", "clicks_first_half", "avg_position_first_half",
                    "sessions_first_half", "content_age_days_at_decision"]

model_data = data.dropna(subset=honest_features + ["is_declining"])
X, y = model_data[honest_features], model_data["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"HONEST auc (feature-window-only): {honest_auc:.3f}")


HONEST auc (feature-window-only): 0.595


**Now spring the trap — add the raw target-window value as a feature:**

`impressions_second_half` is literally the number the label is thresholded from. It should not
be knowable at the decision moment (it's the outcome, not the input), but I'm adding it on
purpose to see what leakage looks like.


In [10]:
leaky_features = honest_features + ["impressions_second_half"]

model_data_leak = data.dropna(subset=leaky_features + ["is_declining"])
X, y = model_data_leak[leaky_features], model_data_leak["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

leaky_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
leaky_auc = roc_auc_score(y_te, leaky_model.predict_proba(X_te)[:, 1])
print(f"LEAKY auc (target value smuggled in as a feature): {leaky_auc:.3f}")
print(f"jump vs honest score: +{leaky_auc - honest_auc:.3f}")


LEAKY auc (target value smuggled in as a feature): 1.000
jump vs honest score: +0.405


**Delete the leak, keep the honest number.**

The jump above is the tell: `impressions_second_half` isn't a real feature, it's the answer
wearing a feature's clothes — because `is_declining` is defined directly from it. I remove it
and carry the honest, feature-window-only score forward as the number I actually trust.


In [11]:
print(f"Score I'm keeping for this lane: HONEST auc = {honest_auc:.3f}")
print("impressions_second_half dropped from the feature set going forward.")


Score I'm keeping for this lane: HONEST auc = 0.595
impressions_second_half dropped from the feature set going forward.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation:** this is an unbalanced panel — clients don't all have the same amount
of history, and rows before a client's GA4 tracking start carry `ga4_data_available = FALSE`
(search-only, no engagement signal). If I don't check `dim_clients.gsc_data_start` /
`ga4_data_start` before defining a window, I risk quietly treating "this client wasn't tracked
yet" as "this page had no traffic" — which would bias any feature or label built from GA4
sessions toward clients with longer history. Query 3 above is exactly the check that catches this
for the March slice; the same check needs to be repeated for any other month I use later.


In [12]:
limit_check = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    WHERE gsc_data_start > DATE '{MONTH_START}' OR ga4_data_start > DATE '{MONTH_START}'
    ORDER BY gsc_data_start
""").df()

print(f"{len(limit_check)} clients whose tracking started after this slice's month began")
limit_check.head()


27 clients whose tracking started after this slice's month began


,client_hash_id,gsc_data_start,ga4_data_start
0,client_73cda7b4e4f265ea,2025-02-11,2026-03-24
1,client_fef1a8f436438636,2025-03-11,2026-03-06
2,client_3ffa76342f366962,2025-10-11,2026-03-11
3,client_cd12bcfd98942aa1,2025-10-20,2026-03-03
4,client_157ffe4d4a595515,2026-02-19,2026-03-09


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.